# 01. 데이터 탐색 (EDA)

**2조 — 백혈구(WBC) 현미경 이미지 분류** / 데이터: Kaggle `paultimothymooney/blood-cells`

### 전체 진행 순서

```
01 데이터탐색  →  02 베이스라인  →  03 이미지전처리  →  04 모델비교·선정
                        ↑                                      ↓
                        └──────  05 하이퍼파라미터 최적화  ←────┘
                                        ↓
                     06 최종학습·분류  →  07 CAM  →  08 가설검정
```

02~05 는 **한 바퀴 돌고 다시 돌아오는 반복 구간**이다. 한 번에 끝내지 않는다.
모든 실험 결과는 `results/runs.csv` 에 자동으로 쌓이므로, 반복해도 기록이 남는다.

---

### 이 노트북에서 하는 일

학습을 시작하기 전에 **데이터가 무엇인지 정확히 아는 것**. 여기서 확인한 사실이
뒤 노트북의 전처리·모델·가설을 전부 결정한다.

| 절 | 확인할 것 | 어디로 이어지나 |
|---|---|---|
| 1-1 | 폴더 구조와 클래스별 장수 | 라벨 순서, 불균형 여부 |
| 1-2 | 클래스가 4개인가 5개인가 | 문제 정의 |
| 1-3 | 실제 이미지 관찰 | **03 이미지전처리** |
| 1-4 | 검은 패딩 비율 측정 | **03 이미지전처리** (자르기 근거) |
| 1-5 | TRAIN/TEST 누수 점검 | **08 가설검정 3** |
| 1-6 | 외부 검증셋(원본 366장) 생성 | **08 가설검정 3** |

In [ ]:
import os, sys, glob, collections, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import wbc
wbc.use_korean_font()          # 그래프 한글 깨짐 방지 (윈도우: 맑은 고딕)

print('작업 폴더 :', os.getcwd())
print('연산 장치 :', wbc.device)
print('데이터 경로:', wbc.DATA_DIR, '->', os.path.isdir(wbc.DATA_DIR))
print('원본 경로  :', wbc.ORIG_DIR, '->', os.path.isdir(wbc.ORIG_DIR))

## 1-1. 폴더 구조가 곧 라벨

수업 5장의 `ImageFolder` 규칙 그대로다. **하위 폴더 이름이 클래스**이고
**알파벳 순으로 번호가 매겨진다.** 이 순서를 여기서 확정하지 않으면
혼동행렬과 CAM 해석이 통째로 어긋난다.

In [ ]:
rows = []
for split in ['TRAIN', 'TEST']:
    for c in wbc.CLASS_NAMES:
        d = os.path.join(wbc.DATA_DIR, split, c)
        rows.append(dict(split=split, cls=c, n=len(os.listdir(d))))
df = pd.DataFrame(rows).pivot(index='cls', columns='split', values='n')
df['label'] = [wbc.CLASS_NAMES.index(c) for c in df.index]
df['TRAIN 비율(%)'] = (df['TRAIN'] / df['TRAIN'].sum() * 100).round(2)
display(df[['label', 'TRAIN', 'TEST', 'TRAIN 비율(%)']])
print('TRAIN 합계', df['TRAIN'].sum(), '/ TEST 합계', df['TEST'].sum())

### 읽는 법

- 라벨 번호: **EOSINOPHIL 0, LYMPHOCYTE 1, MONOCYTE 2, NEUTROPHIL 3** (알파벳 순)
- **네 클래스가 거의 정확히 25%씩** — 4장 헬멧(21:79), 5장 X-ray(26:74)와 달리 **불균형이 없다.**
  - `pos_weight` / 클래스 가중치가 필요 없다
  - 정확도를 주지표로 써도 왜곡이 적다. 다만 한 클래스만 망가지는 걸 놓치지 않으려고 **macro-F1** 을 함께 본다
- **무작위로 찍으면 0.25.** 모든 성능은 이 기준선 위에서 읽는다.

## 1-2. 클래스는 4개인가 5개인가 — BASOPHIL 문제

백혈구는 호중구·림프구·단핵구·호산구·**호염기구(BASOPHIL)** 의 5종이 맞다.
그런데 학습에 쓸 `dataset2-master` 에는 폴더가 4개뿐이다. 원본 라벨을 직접 세어 확인한다.

In [ ]:
lab = pd.read_csv(os.path.join(wbc.ORIG_DIR, 'labels.csv'))
if lab.shape[1] == 3: lab.columns = ['idx', 'Image', 'Category']
display(lab['Category'].fillna('(비어있음)').value_counts().to_frame('장수'))

### 결론 — 4클래스로 간다

- `BASOPHIL` 은 **원본 366장 중 3장**뿐이다.
- 3장으로는 학습도, 평가도, 통계적 검정도 불가능하다.
  검증셋에 1장 들어가면 그 클래스 재현율은 0 아니면 1 로만 나온다.
- 그래서 데이터셋 제작자도 증강본에서 BASOPHIL 을 뺐다.

**문제 정의: 4클래스 분류.** 보고서에는 "5종 중 BASOPHIL 은 표본 3장으로
학습·평가가 불가능하여 제외" 라고 명시한다. **데이터의 한계이지 우리 선택의 실수가 아니다.**

> 라벨에 `NEUTROPHIL, EOSINOPHIL` 처럼 두 종류가 적힌 원본이 12장 있다.
> 한 장에 백혈구가 둘 이상 찍힌 경우로, 1-6 의 외부 검증셋에서는 제외한다.

## 1-3. 실제 이미지를 본다

**모델을 고르기 전에 눈으로 본다.** 여기서 전처리 설계가 갈린다.

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(15, 9))
rng = np.random.RandomState(0)
for r, c in enumerate(wbc.CLASS_NAMES):
    d = os.path.join(wbc.DATA_DIR, 'TRAIN', c)
    fs = sorted(os.listdir(d))
    for k, f in enumerate(rng.choice(fs, min(5, len(fs)), replace=False)):
        axes[r, k].imshow(Image.open(os.path.join(d, f))); axes[r, k].axis('off')
    axes[r, 0].set_title(f'{c} (label {r})', loc='left', fontsize=11)
plt.tight_layout(); plt.show()

### 눈으로 확인되는 것 5가지 → 전부 03 노트북의 가설이 된다

1. **이미 증강된 데이터다.** 사진이 기울어져 있고 모서리에 검은 삼각형(회전 패딩)이 보인다.
   원본 366장을 회전·이동·확대해 12,444장으로 불린 것이다.
   → *회전 증강을 또 크게 주면 패딩 위에 패딩을 쌓는 꼴이다.*
2. **판단 근거는 가운데 진한 보라색 세포 하나.** 주변 분홍 원반은 적혈구로, 모든 클래스에 공통이다.
   → *배경을 보고 맞히면 안 된다. **07 CAM** 에서 확인한다.*
3. **클래스별 생김새**
   | 클래스 | 핵 | 세포질 |
   |---|---|---|
   | EOSINOPHIL | 두 덩이(이엽) | 분홍·주황 과립 |
   | LYMPHOCYTE | 작고 둥근 진한 핵 | 거의 없음 |
   | MONOCYTE | 콩팥·말굽 모양, 가장 큼 | 넓고 회청색 |
   | NEUTROPHIL | 3~5 덩이로 잘게 갈라짐 | 옅음 |

   → *형태가 핵심 신호, 색은 보조 신호(호산구 과립). 그래서 **색 증강은 약하게만**.*
4. **세포가 항상 가운데 있지 않다.** 강한 자르기는 세포를 잘라낼 위험이 있다.
5. **모든 이미지가 320×240** 으로 크기가 일정하다.

## 1-4. 검은 패딩이 얼마나 되나 — 자르기(Crop)의 근거

"패딩이 있으니 잘라내자"는 감이고, **잘라낼 비율은 숫자로 정한다.**
거의 검은 픽셀의 비율을 재고, 중앙 몇 %를 남기면 그게 사라지는지 본다.

In [ ]:
def dark_ratio(arr, thr=25):
    """거의 검은 픽셀(회전 패딩)의 비율"""
    return float((arr.max(axis=2) < thr).mean())

sample = []
rng = np.random.RandomState(1)
for c in wbc.CLASS_NAMES:
    d = os.path.join(wbc.DATA_DIR, 'TRAIN', c)
    files = sorted(os.listdir(d))
    for f in rng.choice(files, min(150, len(files)), replace=False):
        sample.append(np.asarray(Image.open(os.path.join(d, f)).convert('RGB')))

full = np.array([dark_ratio(a) for a in sample])
ratios = {}
for r in [0.9, 0.8, 0.7, 0.6]:
    h, w = int(240 * r), int(320 * r)
    y, x = (240 - h) // 2, (320 - w) // 2
    ratios[r] = np.array([dark_ratio(a[y:y+h, x:x+w]) for a in sample])

print(f'원본(자르지 않음) : 검은픽셀 평균 {full.mean()*100:5.2f}%   1% 넘는 이미지 {(full>0.01).mean()*100:5.1f}%')
for r, v in ratios.items():
    print(f'중앙 {int(r*100)}% 만 사용 : 검은픽셀 평균 {v.mean()*100:5.2f}%   1% 넘는 이미지 {(v>0.01).mean()*100:5.1f}%')

In [ ]:
plt.figure(figsize=(6.5, 3.6))
plt.hist(full * 100, bins=40, alpha=.6, label='원본')
plt.hist(ratios[0.8] * 100, bins=40, alpha=.6, label='중앙 80%')
plt.xlabel('이미지에서 검은 픽셀이 차지하는 비율 (%)'); plt.ylabel('이미지 수')
plt.title('회전 패딩(검은 모서리)의 양'); plt.legend(); plt.grid(alpha=.3); plt.show()

### 해석

중앙 80%만 남기면 검은 패딩이 크게 줄어든다. 다만 1-3 에서 봤듯 세포가 가장자리에 있는 경우가 있어
**세게 자르면 정작 봐야 할 세포를 잘라낸다.**

그래서 자르기는 "당연히 좋은 것"이 아니라 **03 노트북에서 실험으로 검증할 가설**로 남긴다.

## 1-5. TRAIN / TEST 누수 점검 — 이 테스트 점수를 믿어도 되나

이 데이터셋은 **원본 366장을 증강해 12,444장으로 불린 것**이다. 그러면 의심할 것이 생긴다.

> 같은 원본에서 나온 사진이 TRAIN 에도 TEST 에도 들어 있으면?
> 모델은 새 데이터를 맞히는 게 아니라 **본 적 있는 사진의 회전본**을 맞히는 셈이고,
> 테스트 정확도는 실제 성능보다 부풀려진다.
> (5장에서 test 경로에 train 이 들어갔던 사고와 성격이 같다)

1단계로 **픽셀 단위 완전 중복**을 확인한다. 32×32 로 줄여 코사인 유사도를 잰다.

In [ ]:
def thumb_vec(path, s=32):
    im = Image.open(path); im.draft('L', (64, 48))
    a = np.asarray(im.convert('L').resize((s, s)), dtype=np.float32).ravel()
    a = (a - a.mean()) / (a.std() + 1e-6)
    return a / (np.linalg.norm(a) + 1e-9)

V = {}
for split in ['TRAIN', 'TEST']:
    for c in wbc.CLASS_NAMES:
        d = os.path.join(wbc.DATA_DIR, split, c)
        V[(split, c)] = np.stack([thumb_vec(os.path.join(d, f)) for f in sorted(os.listdir(d))])
    print(split, '완료')

In [ ]:
print('TEST 각 장에 대해 가장 닮은 TRAIN 이미지와의 유사도 (같은 클래스 안에서)')
for c in wbc.CLASS_NAMES:
    A, B = V[('TRAIN', c)], V[('TEST', c)]
    best = np.concatenate([(B[i:i+256] @ A.T).max(1) for i in range(0, len(B), 256)])
    print(f'  {c:11s} 중앙값 {np.median(best):.3f} | >0.99 {(best>0.99).mean()*100:4.1f}% | >0.95 {(best>0.95).mean()*100:4.1f}%')

# 대조군 — 이 지표가 클래스를 구분하는 힘이 있는지 확인한다
A = np.concatenate([V[('TRAIN', c)] for c in wbc.CLASS_NAMES[1:]])
B = V[('TEST', wbc.CLASS_NAMES[0])]
best = np.concatenate([(B[i:i+256] @ A.T).max(1) for i in range(0, len(B), 256)])
print(f'  [대조군] EOSINOPHIL TEST vs 다른 클래스 TRAIN : 중앙값 {np.median(best):.3f} | >0.95 {(best>0.95).mean()*100:4.1f}%')

### 결론과 한계 (보고서에 그대로 쓸 것)

- **완전히 같은 사진이 TRAIN 과 TEST 에 동시에 있지는 않다** (유사도 0.99 초과가 사실상 0%).
- 그런데 **대조군(다른 클래스)의 유사도가 같은 클래스와 거의 같다.**
  이 축소 썸네일 지표는 **클래스를 구분하는 힘이 없다**는 뜻이고,
  따라서 **회전·이동된 형제 이미지까지 잡아내지는 못한다.**
- 즉 말할 수 있는 것은 *"복사-붙여넣기 수준의 중복은 없다"* 까지다.
  **원본 단위 분리는 보장되지 않는다.**

그래서 테스트 정확도가 아주 높게 나오더라도(예: 99%) 그것을 곧바로 실제 성능이라 주장하지 않는다.
대신 **증강되지 않은 원본 이미지**로 한 번 더 재고, 두 값이 통계적으로 다른지
**08 노트북의 가설검정 3** 에서 검정한다. 이번 프로젝트에서 가장 중요한 방법론적 판단이다.

## 1-6. 외부 검증셋 만들기 — 원본 366장

`dataset-master/JPEGImages` 의 640×480 원본은 **증강 전 이미지**다.
`labels.csv` 를 이용해 `external/<클래스>/` 형식으로 정리한다.

규칙
- 라벨이 하나뿐인 이미지만 (복수 라벨 제외)
- 4개 클래스만 (BASOPHIL 제외)
- 평가할 때는 화각을 맞추려고 **중앙 70% 를 잘라** 쓴다

In [ ]:
EXT_DIR = './external'
lab = pd.read_csv(os.path.join(wbc.ORIG_DIR, 'labels.csv'))
if lab.shape[1] == 3: lab.columns = ['idx', 'Image', 'Category']
lab = lab.dropna(subset=['Category'])
lab['Category'] = lab['Category'].str.strip()
single = lab[lab['Category'].isin(wbc.CLASS_NAMES)]          # 복수 라벨 자동 제외

if os.path.isdir(EXT_DIR): shutil.rmtree(EXT_DIR)
n_copy = collections.Counter()
for _, r in single.iterrows():
    src = os.path.join(wbc.ORIG_DIR, 'JPEGImages', f'BloodImage_{int(r["Image"]):05d}.jpg')
    if not os.path.exists(src): continue
    dst_d = os.path.join(EXT_DIR, r['Category']); os.makedirs(dst_d, exist_ok=True)
    shutil.copy(src, os.path.join(dst_d, os.path.basename(src)))
    n_copy[r['Category']] += 1

print('외부 검증셋 :', dict(n_copy), '총', sum(n_copy.values()), '장')
print('제외된 것   :', len(lab) - len(single), '장 (복수 라벨 / BASOPHIL)')

In [ ]:
# 원본과 증강본의 화각 비교 — 왜 중앙 70% 인지 눈으로 확인
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
o = Image.open(sorted(glob.glob(os.path.join(EXT_DIR, '*', '*.jpg')))[0]).convert('RGB')
w, h = o.size
ax[0].imshow(o); ax[0].set_title(f'원본 {o.size}'); ax[0].axis('off')
ax[1].imshow(o.crop((int(w*.15), int(h*.15), int(w*.85), int(h*.85))))
ax[1].set_title('원본 중앙 70%'); ax[1].axis('off')
d = os.path.join(wbc.DATA_DIR, 'TRAIN', 'NEUTROPHIL')
ax[2].imshow(Image.open(os.path.join(d, sorted(os.listdir(d))[3])))
ax[2].set_title('증강본 (320x240)'); ax[2].axis('off')
plt.tight_layout(); plt.show()

## 01 정리 — 여기서 확정된 것

| 확정 사항 | 근거 | 어디에 쓰이나 |
|---|---|---|
| 4클래스, 라벨 순서 E-L-M-N | 1-1, 1-2 | 전 과정 |
| 클래스 균형 → 가중치 불필요, 지표는 정확도 + macro-F1 | 1-1 | 전 과정 |
| 무작위 기준선 = 0.25 | 1-1 | 성능 해석 |
| 회전 증강이 이미 적용됨 → 추가 회전은 검증 대상 | 1-3, 1-4 | **03 이미지전처리** |
| 판단 근거는 백혈구 핵, 배경 적혈구가 아님 | 1-3 | **07 CAM** |
| 완전중복은 없으나 원본 단위 분리는 보장 안 됨 | 1-5 | **08 가설검정 3** |
| `external/` 생성 완료 | 1-6 | **08 가설검정 3** |

→ 다음: **02_베이스라인.ipynb**